In [1]:
# ============================================================
# 06_layout_feature_extraction_v2 - Cell 1
# Imports, dataset_final labels load, layout function define aur 2 images pe test
# ============================================================

# OpenCV image processing ke liye
import cv2

# NumPy numerical operations ke liye
import numpy as np

# Pandas CSV/table handling ke liye
import pandas as pd

# os module paths ke liye
import os

# time module total time naapne ke liye
import time

# Project, dataset aur images folder ke paths (dataset_final)
PROJECT_DIR = r"C:\Projects\Internship_Learning"
DATASET_DIR = os.path.join(PROJECT_DIR, "dataset_final")
IMAGES_DIR = os.path.join(DATASET_DIR, "images")

# labels.csv load karta hai aur uske filenames ki list banata hai (position-based nahi)
labels_df = pd.read_csv(os.path.join(DATASET_DIR, "labels.csv"))
filenames = labels_df["filename"].tolist()

# Total filenames print karta hai (979 hone chahiye)
print("Total filenames:", len(filenames))


# Layout features nikalne ka function (logic purane notebook 06 jaisa hi hai)
def extract_layout_features(image_path):

    # Image load karta hai
    img = cv2.imread(image_path)

    # Agar image padh nahi payi to None return karta hai
    if img is None:
        return None

    # Image ko 224x224 me resize karta hai (purane notebook jaisa, taaki comparison fair rahe)
    img = cv2.resize(img, (224, 224))

    # Grayscale me convert karta hai
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

    # Height aur width store karta hai
    h, w = gray.shape

    # Threshold 180 pe binary inverse: dark/content regions white ho jaate hain
    _, thresh = cv2.threshold(gray, 180, 255, cv2.THRESH_BINARY_INV)

    # 3x3 kernel banata hai
    kernel = np.ones((3, 3), np.uint8)

    # Dilation (1 iteration) se nearby regions merge karta hai
    thresh = cv2.dilate(thresh, kernel, iterations=1)

    # Outer contours (visual blocks) detect karta hai
    contours, _ = cv2.findContours(thresh, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

    # Area > 100 wale contours hi rakhta hai (noise hatane ke liye)
    valid_contours = [c for c in contours if cv2.contourArea(c) > 100]

    # 1. num_blocks: total visual blocks
    num_blocks = len(valid_contours)

    # 2. avg_block_size: average block area
    avg_block_size = round(float(np.mean([cv2.contourArea(c) for c in valid_contours])), 2) if num_blocks > 0 else 0.0

    # Saare blocks ka total area
    total_block_area = sum(cv2.contourArea(c) for c in valid_contours)

    # 3. block_density: blocks ne image ka kitna % area gher rakha hai
    block_density = round(total_block_area / (h * w) * 100, 2)

    # Left half me kitne blocks hain (bounding box ka x < w/2)
    left_blocks = sum(1 for c in valid_contours if cv2.boundingRect(c)[0] < w // 2)

    # Baaki right me
    right_blocks = num_blocks - left_blocks

    # 4. lr_balance: 0 = balanced, 1 = ek side pe saare blocks
    lr_balance = round(abs(left_blocks - right_blocks) / (num_blocks + 1e-5), 2)

    # Top half me kitne blocks hain (bounding box ka y < h/2)
    top_blocks = sum(1 for c in valid_contours if cv2.boundingRect(c)[1] < h // 2)

    # Baaki bottom me
    bottom_blocks = num_blocks - top_blocks

    # 5. tb_balance: top-bottom balance
    tb_balance = round(abs(top_blocks - bottom_blocks) / (num_blocks + 1e-5), 2)

    # 6. text_like_regions: chhote blocks (area < 500)
    text_like = sum(1 for c in valid_contours if cv2.contourArea(c) < 500)

    # 7. image_like_regions: bade blocks (area > 5000)
    image_like = sum(1 for c in valid_contours if cv2.contourArea(c) > 5000)

    # 8. layout_clutter_score: purana layout clutter formula (naam alag taaki visual clutter_score se na takraye)
    layout_clutter_score = round((block_density * 0.5) + (text_like * 0.5), 2)

    # Saare features dictionary me return karta hai
    return {
        "num_blocks": num_blocks,
        "avg_block_size": avg_block_size,
        "block_density": block_density,
        "lr_balance": lr_balance,
        "tb_balance": tb_balance,
        "text_like_regions": text_like,
        "image_like_regions": image_like,
        "layout_clutter_score": layout_clutter_score,
    }


# Quick test: pehli 2 images pe function chalata hai
for fname in filenames[:2]:
    print(fname)
    print(extract_layout_features(os.path.join(IMAGES_DIR, fname)))
    print()

Total filenames: 979
old_0001.png
{'num_blocks': 16, 'avg_block_size': 2039.47, 'block_density': 65.03, 'lr_balance': 0.37, 'tb_balance': 0.5, 'text_like_regions': 9, 'image_like_regions': 1, 'layout_clutter_score': 37.02}

old_0002.png
{'num_blocks': 11, 'avg_block_size': 1954.32, 'block_density': 42.84, 'lr_balance': 0.09, 'tb_balance': 0.45, 'text_like_regions': 8, 'image_like_regions': 1, 'layout_clutter_score': 25.42}



In [2]:
# ============================================================
# 06_layout_feature_extraction_v2 - Cell 2
# Saari 979 images pe layout features, layout_features.csv save, features_all.csv merge
# ============================================================

# Har image ke layout features yahan jama honge
layout_rows = []

# Failed images ki list
layout_failed = []

# Timer start
start_time = time.time()

# labels.csv ke filenames pe loop
for i, fname in enumerate(filenames, start=1):

    # Image ka full path
    path = os.path.join(IMAGES_DIR, fname)

    # Try-except taaki ek image ke error se loop na ruke
    try:
        lf = extract_layout_features(path)
    except Exception as e:
        print("Error:", fname, e)
        lf = None

    # None aaye to failed me daalta hai
    if lf is None:
        layout_failed.append(fname)
    else:
        # filename jodta hai taaki merge ho sake
        lf["filename"] = fname
        layout_rows.append(lf)

    # Har 100 images pe progress print
    if i % 100 == 0:
        print(f"{i}/{len(filenames)} done | {time.time() - start_time:.1f} sec")

# DataFrame banata hai aur filename ko sabse aage laata hai
layout_df = pd.DataFrame(layout_rows)
layout_df = layout_df[["filename"] + [c for c in layout_df.columns if c != "filename"]]

# layout_features.csv save karta hai
LAYOUT_CSV = os.path.join(DATASET_DIR, "layout_features.csv")
layout_df.to_csv(LAYOUT_CSV, index=False)

# Summary print karta hai
print(f"\nTotal time: {time.time() - start_time:.1f} sec")
print("Processed:", len(layout_df), "| Failed:", len(layout_failed), layout_failed[:10])
print("Saved:", LAYOUT_CSV)

# Null check (sab 0 hone chahiye)
print("\nNull values:", int(layout_df.isnull().sum().sum()))

# ---------- features.csv (visual+text) ke saath merge ----------

# Purani features.csv load karta hai (979 rows, 18 columns)
features_df = pd.read_csv(os.path.join(DATASET_DIR, "features.csv"))

# filename pe merge karta hai (layout columns judte hain)
features_all = features_df.merge(layout_df, on="filename", how="inner")

# Checks: 979 rows aur filenames unique
assert len(features_all) == 979, f"Rows mismatch: {len(features_all)}"
assert features_all["filename"].is_unique

# label, score, source columns ko end me laata hai (features pehle, meta baad me)
meta_cols = ["label", "score", "source"]
feature_cols_all = [c for c in features_all.columns if c not in ["filename"] + meta_cols]
features_all = features_all[["filename"] + feature_cols_all + meta_cols]

# features_all.csv save karta hai (features.csv untouched rehti hai)
ALL_CSV = os.path.join(DATASET_DIR, "features_all.csv")
features_all.to_csv(ALL_CSV, index=False)

# Shape aur feature count print karta hai (979 rows, 22 features + 4 = 26 columns aana chahiye)
print("\nSaved:", ALL_CSV)
print("Shape:", features_all.shape)
print("Feature columns:", len(feature_cols_all))

# Display settings
pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 30)

# Label-wise mean sirf layout columns ke liye
layout_cols = [c for c in layout_df.columns if c != "filename"]
print("\nLabel-wise mean (layout features):")
print(features_all.groupby("label")[layout_cols].mean().round(3).T)

# Source-wise mean sirf layout columns ke liye (source bias check)
print("\nSource-wise mean (layout features):")
print(features_all.groupby("source")[layout_cols].mean().round(3).T)

100/979 done | 3.0 sec
200/979 done | 5.9 sec
300/979 done | 8.7 sec
400/979 done | 11.5 sec
500/979 done | 14.2 sec
600/979 done | 16.9 sec
700/979 done | 19.6 sec
800/979 done | 22.2 sec
900/979 done | 25.0 sec

Total time: 27.3 sec
Processed: 979 | Failed: 0 []
Saved: C:\Projects\Internship_Learning\dataset_final\layout_features.csv

Null values: 0

Saved: C:\Projects\Internship_Learning\dataset_final\features_all.csv
Shape: (979, 26)
Feature columns: 22

Label-wise mean (layout features):
label                   average       good       poor
num_blocks                7.702      5.771      8.180
avg_block_size        10205.384  18745.190  11446.880
block_density            49.033     60.056     51.557
lr_balance                0.567      0.684      0.600
tb_balance                0.500      0.641      0.510
text_like_regions         5.132      3.574      5.401
image_like_regions        0.840      0.878      0.880
layout_clutter_score     27.082     31.814     28.478

Source-wise mea